In [ ]:
import os
import requests
import gzip
import shutil
from dotenv import load_dotenv

load_dotenv("secrets.env")  # Solo si necesitas API keys

def download_file(url, local_path):
    """Descarga robusta con streaming."""
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise RuntimeError(f"Error descargando {url}")

    with open(local_path, "wb") as f:
        shutil.copyfileobj(r.raw, f)

    if os.path.getsize(local_path) < 1000:
        raise RuntimeError(f"Archivo sospechosamente pequeño: {local_path}")

    return local_path


def decompress_gz(path_gz, path_txt):
    """Descomprime .gz a .txt"""
    with gzip.open(path_gz, "rb") as f_in:
        with open(path_txt, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    return path_txt


def load_language_dataset(idioma):
    idioma = idioma.lower()
    if idioma not in ["gallego", "asturiano", "aranes"]:
        raise ValueError("Idioma no soportado")

    # Inicializa con Tatoeba activado
    ds = LanguageDataset(idioma, initializeTatoeba=True)

    # -----------------------------
    # 1. GALLEGO → NOS Corpus
    # -----------------------------
    if idioma == "gallego":
        print("Descargando NOS Corpus (gallego)...")

        nos_url = "https://zenodo.org/records/10687642/files/nos_corpus_v1.0.0.txt?download=1"
        nos_local = "nos_corpus_gl.txt"

        download_file(nos_url, nos_local)
        ds.read_local_file(".", nos_local)
        print("NOS Corpus cargado correctamente")

    # -----------------------------
    # 2. OPUS-NLLB monolingüe
    # -----------------------------
    opus_map = {
        "gallego": "gl",      # aunque no lo pediste, lo dejo preparado
        "asturiano": "ast",
        "aranes": "oc"
    }

    opus_code = opus_map[idioma]
    opus_url = f"https://object.pouta.csc.fi/OPUS-NLLB/v1/mono/{opus_code}.txt.gz"

    print(f"Descargando OPUS-NLLB monolingüe ({opus_code})…")

    gz_path = f"opus_{opus_code}.txt.gz"
    txt_path = f"opus_{opus_code}.txt"

    download_file(opus_url, gz_path)
    decompress_gz(gz_path, txt_path)

    ds.read_local_file(".", txt_path)
    print("OPUS-NLLB cargado correctamente")

    return ds


In [ ]:
hf_dataset = load_language_dataset("gallego").hf_dataset

splits = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
test_dataset  = splits["test"]
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)
